## Local IndicTrans2 Inference

This notebook acts as an interactive inference pipeline to load your local compiled checkpoints or remote Hub checkpoints and translate custom phrases.

**Prerequisites:**
- Ensure you have selected your project's `.venv` as the Jupyter kernel.
- Ensure `./checkpoints/` has your finetuned model, or change `adapter_path` to your pushed Hugging Face repo.

In [ ]:
# Cell 1: Setup Workspace
import os
import sys
import torch

# Move to project root if running from inside notebooks/
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print("Current Working Directory:", os.getcwd())

sys.path.append(os.getcwd())
from pipeline.model.inference.indictrans import IndicTransTranslator
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

In [ ]:
# Cell 2: Load the Base Model & Tokenizer
base_model_name = "ai4bharat/indictrans2-en-indic-dist-200M"

print(f"Loading tokenizer from: {base_model_name}")
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)

# Use QLoRA (4-bit quantization) to fit larger batches natively if memory is tight
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading base model in 4-bit: {base_model_name}")
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

In [ ]:
# Cell 3: Apply the PEFT Adapter and setup Translator Core

# Set your local checkpoint path or HuggingFace repo link (e.g., "Firoj112/indictrans2-en-npi-mai-finetuned")
adapter_path = "./checkpoints" # or "HF_REPO_NAME"

print(f"Applying Adapter from: {adapter_path}")
model = PeftModel.from_pretrained(base_model, adapter_path)

# Default inference configuration (can be tweaked)
inference_cfg = {
    "evaluation": {
        "num_beams": 5,
        "max_length": 256,
        "length_penalty": 1.0,
        "no_repeat_ngram_size": 3
    }
}

translator = IndicTransTranslator(model, tokenizer, inference_cfg)
print("Translator ready!")

In [ ]:
# Cell 4: Single phrase generation
source_text = "This is a local inference test indicating whether the custom translation works properly."
src_lang = "eng_Latn"
tgt_lang = "npi_Deva"  # Options: npi_Deva (Nepali), mai_Deva (Maithili)

result = translator.translate(source_text, src_lang=src_lang, tgt_lang=tgt_lang)
print(f"Source ({src_lang}): {source_text}")
print(f"Target ({tgt_lang}): {result}")

In [ ]:
# Cell 5: Batch inference performance
sentences = [
    "Nepal is a beautiful country.",
    "I have to study machine learning today.",
    "We are preparing the data for the training later this week."
]

results = translator.translate_batch(sentences, src_lang="eng_Latn", tgt_lang="npi_Deva", batch_size=4)

for src, trg in zip(sentences, results):
    print(f"EN: {src}")
    print(f"NPI: {trg}\n")